# 네컷사진 판별 - 단층 퍼셉트론 (SLP)

이 노트북은 단층 퍼셉트론을 사용하여 네컷사진을 판별하는 방법을 시연합니다.

## 1. 라이브러리 임포트

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# src 모듈 경로 추가
sys.path.append(str(Path.cwd().parent))

from src.preprocessing import ImagePreprocessor
from src.model import SingleLayerPerceptron

## 2. 데이터 준비

샘플 데이터를 생성합니다. 실제 사용 시에는 `data/raw/fourcut`과 `data/raw/non_fourcut` 폴더에 실제 이미지를 넣어주세요.

In [ ]:
# 전처리기 초기화
preprocessor = ImagePreprocessor(
    target_size=(64, 64),
    grayscale=True,
    normalize=True
)

# 샘플 데이터 생성 (예제용)
# 실제로는 이미지 파일을 로드해야 합니다
np.random.seed(42)
n_samples = 200
n_features = 64 * 64  # 64x64 그레이스케일 이미지

# 네컷사진 패턴 (밝은 영역이 4개 구역으로 나뉨)
X_fourcut = np.random.rand(n_samples // 2, n_features) * 0.3 + 0.5

# 일반 사진 패턴 (더 불규칙함)
X_non_fourcut = np.random.rand(n_samples // 2, n_features) * 0.5 + 0.2

# 데이터 결합
X = np.vstack([X_fourcut, X_non_fourcut])
y = np.hstack([np.ones(n_samples // 2), np.zeros(n_samples // 2)])

# 데이터 섞기
indices = np.random.permutation(n_samples)
X = X[indices]
y = y[indices]

print(f"데이터 형태: {X.shape}")
print(f"레이블 형태: {y.shape}")
print(f"네컷사진: {np.sum(y == 1)}개")
print(f"일반 사진: {np.sum(y == 0)}개")

## 3. 데이터 분할

In [ ]:
# 학습/테스트 세트 분할
test_ratio = 0.2
test_size = int(n_samples * test_ratio)

X_train = X[test_size:]
X_test = X[:test_size]
y_train = y[test_size:]
y_test = y[:test_size]

print(f"학습 세트: {X_train.shape[0]}개")
print(f"테스트 세트: {X_test.shape[0]}개")

## 4. 모델 학습

In [ ]:
# 모델 초기화
model = SingleLayerPerceptron(
    input_size=n_features,
    learning_rate=0.01,
    epochs=100,
    random_state=42
)

# 학습
model.fit(X_train, y_train, verbose=True)

## 5. 학습 곡선 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 손실 곡선
axes[0].plot(model.loss_history)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('학습 손실 곡선')
axes[0].grid(True)

# 정확도 곡선
axes[1].plot(model.accuracy_history)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('학습 정확도 곡선')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. 모델 평가

In [ ]:
# 학습 세트 평가
train_metrics = model.evaluate(X_train, y_train)
print("[학습 세트]")
print(f"손실: {train_metrics['loss']:.4f}")
print(f"정확도: {train_metrics['accuracy']:.4f}")

# 테스트 세트 평가
test_metrics = model.evaluate(X_test, y_test)
print("\n[테스트 세트]")
print(f"손실: {test_metrics['loss']:.4f}")
print(f"정확도: {test_metrics['accuracy']:.4f}")

## 7. 예측 예제

In [ ]:
# 테스트 세트에서 몇 개 샘플 예측
sample_indices = np.random.choice(len(X_test), 5, replace=False)
X_samples = X_test[sample_indices]
y_samples = y_test[sample_indices]

predictions = model.predict(X_samples)
probabilities = model.predict_proba(X_samples)

print("예측 결과:")
print("-" * 60)
for i in range(len(sample_indices)):
    true_label = "네컷사진" if y_samples[i] == 1 else "일반 사진"
    pred_label = "네컷사진" if predictions[i] == 1 else "일반 사진"
    print(f"샘플 {i+1}: 실제={true_label:10s}, 예측={pred_label:10s}, 확률={probabilities[i]:.4f}")

## 8. 모델 저장 및 로드

In [ ]:
# 모델 저장model_path = Path.cwd().parent / 'data' / 'processed' / 'model_weights.npz'model_path.parent.mkdir(parents=True, exist_ok=True)  # Create parent directories if they don't existmodel.save_weights(str(model_path))# 새 모델 생성 및 가중치 로드new_model = SingleLayerPerceptron(input_size=n_features)new_model.load_weights(str(model_path))# 로드된 모델로 평가loaded_metrics = new_model.evaluate(X_test, y_test)print("\n[로드된 모델 평가]")print(f"손실: {loaded_metrics['loss']:.4f}")print(f"정확도: {loaded_metrics['accuracy']:.4f}")